# Knee MRI Multilabel Classification — Experimentation Notebook

This unified notebook covers **Phase 1 (Data Integrity Audit)** and **Phase 2 (Gemini / Clinical NLP Label Re-derivation from Reports)**.

### Objectives:
1. **Git Sync:** Clone or pull the latest code from [https://github.com/Prashanf/Rsna_KneeAbnormality_Detection](https://github.com/Prashanf/Rsna_KneeAbnormality_Detection) directly into the Kaggle environment.
2. **Data Audit (Phase 1):** Validate disk vs CSV parity, series counts, slice counts, DICOM transfer syntax decodability, and produce `series_index.parquet`.
3. **Missing Label Analysis:** Inspect `train.csv` to identify studies lacking condition labels (~98.7% missing in raw data).
4. **Label Re-derivation (Phase 2):** Parse free-text radiology `Report` strings using **Gemini API** (or fallback local 9-language clinical NegEx engine), cache incrementally to disk, and produce clean training targets in `clean_train.csv` and `train_labels_llm.csv`.

In [ ]:
# --- Cell 1: Git Sync & Dependency Check ---
import os, sys, subprocess
from pathlib import Path

# Handle optional private GitHub PAT via Kaggle Secrets (Add-ons -> Secrets -> GITHUB_PAT)
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_PAT")
    REPO_URL = f"https://{token}@github.com/Prashanf/Rsna_KneeAbnormality_Detection.git"
except Exception:
    REPO_URL = "https://github.com/Prashanf/Rsna_KneeAbnormality_Detection.git"

REPO_DIR = "/kaggle/working/repo"

# Check if running on Kaggle
if os.path.exists("/kaggle/working"):
    if not os.path.exists(REPO_DIR):
        print(f"Cloning repo from {REPO_URL} into {REPO_DIR}...")
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    else:
        print(f"Pulling latest changes in {REPO_DIR}...")
        subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

    # Log current commit hash
    commit = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
                            capture_output=True, text=True).stdout.strip()
    print(f"✓ Repository synced at commit: {commit}")

    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)
else:
    local_root = str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd())
    print(f"Running in local workspace: {local_root}")
    if local_root not in sys.path:
        sys.path.insert(0, local_root)

# Ensure DICOM decoders and google-genai are installed
try:
    import pydicom, pylibjpeg
except ImportError:
    print("Installing pydicom and decoders...")
    !pip install -q pydicom pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg

try:
    import google.genai
except ImportError:
    print("Installing google-genai...")
    !pip install -q google-genai


In [ ]:
# --- Cell 2: Purge module cache & load consolidated imports ---
# Purging ensures re-running Cell 2 after git pull immediately reflects updated scripts
for mod in list(sys.modules):
    if mod.startswith("src.") or mod.startswith("environment."):
        del sys.modules[mod]

from environment.imports import *
print("✓ Imported all repo modules and dependencies from environment.imports")

In [ ]:
# --- Cell 3: Resolve dataset paths ---
def resolve_paths():
    kaggle_matches = list(Path('/kaggle/input').glob('*knee*')) + list(Path('/kaggle/input').glob('*rsna*'))
    if kaggle_matches:
        raw_dir = kaggle_matches[0]
        processed_dir = Path('/kaggle/working/data/processed')
        print(f"Kaggle dataset detected: {raw_dir}")
        return raw_dir, processed_dir

    for candidate in [Path('data/raw'), Path('../data/raw')]:
        if candidate.exists():
            return candidate.resolve(), candidate.parent.resolve() / 'processed'

    return Path('data/raw'), Path('data/processed')

RAW_DIR, PROCESSED_DIR = resolve_paths()
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print(f"RAW_DIR:       {RAW_DIR}")
print(f"PROCESSED_DIR: {PROCESSED_DIR}")

--- 
## Phase 1: Data Structure & Integrity Audit

In [ ]:
# --- Cell 4: Run Data Integrity Audit ---
report = run_integrity_audit(
    raw_dir=RAW_DIR,
    processed_dir=PROCESSED_DIR,
    sample_slices=2,
)

In [ ]:
# --- Cell 5: Visualize Audit Distributions ---
import matplotlib.pyplot as plt

report_path = PROCESSED_DIR / 'integrity_report.json'
with open(report_path) as f:
    data = json.load(f)

dists = data.get('distributions', {})
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Series per study distribution
sps_counts = dists.get('series_per_study', {}).get('counts_distribution', {})
if sps_counts:
    axes[0, 0].bar([str(k) for k in sps_counts.keys()], list(sps_counts.values()), color='#2b5c8f')
    axes[0, 0].set_title('Series Count per Study')
    axes[0, 0].set_xlabel('Number of Series')
    axes[0, 0].set_ylabel('Number of Studies')
    axes[0, 0].grid(axis='y', linestyle='--', alpha=0.6)

# 2. Anatomical planes distribution
planes = dists.get('anatomical_planes', {})
if planes:
    axes[0, 1].bar(list(planes.keys()), list(planes.values()), color='#d95f02')
    axes[0, 1].set_title('Anatomical Planes Distribution')
    axes[0, 1].set_xlabel('Plane')
    axes[0, 1].set_ylabel('Count')
    axes[0, 1].grid(axis='y', linestyle='--', alpha=0.6)

# 3. Contrast attributes (Fluid Sensitivity / Fat Sat)
fluid = dists.get('fluid_sensitive', {})
fat = dists.get('fat_suppression', {})
categories = ['Fluid Sens (1)', 'Fluid Sens (0)', 'Fat Sat (1)', 'Fat Sat (0)']
vals = [
    fluid.get('1', fluid.get(1, 0)), fluid.get('0', fluid.get(0, 0)),
    fat.get('1', fat.get(1, 0)), fat.get('0', fat.get(0, 0))
]
axes[1, 0].bar(categories, vals, color='#2ca02c')
axes[1, 0].set_title('Series Protocol Contrast Attributes')
axes[1, 0].set_ylabel('Count')
axes[1, 0].grid(axis='y', linestyle='--', alpha=0.6)

# 4. Target label positive prevalence in train.csv
labels_prev = dists.get('label_prevalence_raw_train_csv', {})
if labels_prev:
    pos_rates = {k: v['positive_rate'] for k, v in labels_prev.items() if v['positive_rate'] is not None}
    axes[1, 1].barh(list(pos_rates.keys()), [v * 100 for v in pos_rates.values()], color='#7570b3')
    axes[1, 1].set_title('Train.csv Label Positive Prevalence (%)')
    axes[1, 1].set_xlabel('Positive Rate (%)')
    axes[1, 1].grid(axis='x', linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
# --- Cell 6: Inspect generated series_index.parquet ---
parquet_path = PROCESSED_DIR / 'series_index.parquet'
if parquet_path.exists():
    df_series = pd.read_parquet(parquet_path)
    print(f"✓ Generated series index: {len(df_series)} entries saved to {parquet_path}")
    display(df_series.head(10))
else:
    print(f"Notice: {parquet_path} not found.")

--- 
## Phase 2: Missing Label Inspection & LLM Re-derivation

In [ ]:
# --- Cell 7: Inspect Missing Labels in train.csv ---
train_df = pd.read_csv(RAW_DIR / 'train.csv')
print(f"Total training studies: {len(train_df)}")

# Check for missing/null values across the 12 target columns
label_null_counts = {}
for label in TARGET_LABELS:
    if label in train_df.columns:
        label_null_counts[label] = train_df[label].isna().sum()

print("Missing (NaN) label count per finding in train.csv:")
for lbl, cnt in label_null_counts.items():
    print(f"  {lbl:<18}: {cnt} missing ({cnt/len(train_df)*100:.1f}%)")

# Count studies with completely missing labels vs partial vs full
present_cols = [c for c in TARGET_LABELS if c in train_df.columns]
missing_per_row = train_df[present_cols].isna().sum(axis=1)

all_missing = (missing_per_row == len(present_cols)).sum()
partially_missing = ((missing_per_row > 0) & (missing_per_row < len(present_cols))).sum()
fully_labeled = (missing_per_row == 0).sum()

print(f"\nStudy-Level Label Completeness Breakdown:")
print(f"  Fully labeled studies:     {fully_labeled} ({fully_labeled/len(train_df)*100:.1f}%)")
print(f"  Partially labeled studies: {partially_missing} ({partially_missing/len(train_df)*100:.1f}%)")
print(f"  Completely unlabelled:     {all_missing} ({all_missing/len(train_df)*100:.1f}%)")

# Display sample unlabelled studies with their reports
unlabelled_df = train_df[missing_per_row > 0]
if len(unlabelled_df) > 0:
    print(f"\nSample studies with missing labels (Report available):")
    for _, sample_row in unlabelled_df.head(3).iterrows():
        print(f"\n--- Study: {sample_row['StudyInstanceUID']} ---")
        print(f"Report: {str(sample_row.get('Report', ''))[:300]}...")

In [ ]:
# --- Cell 8: Setup LLM Credentials & Check Resumption Progress ---
gemini_key = None
anthropic_key = None

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    try:
        gemini_key = secrets.get_secret("GEMINI_API_KEY")
    except Exception:
        pass
    if not gemini_key:
        try:
            gemini_key = secrets.get_secret("GOOGLE_API_KEY")
        except Exception:
            pass
    try:
        anthropic_key = secrets.get_secret("ANTHROPIC_API_KEY")
    except Exception:
        pass
except Exception:
    pass

gemini_key = gemini_key or os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
anthropic_key = anthropic_key or os.environ.get("ANTHROPIC_API_KEY")

if gemini_key:
    print("✓ Detected GEMINI_API_KEY (Gemini LLM mode enabled).")
elif anthropic_key:
    print("✓ Detected ANTHROPIC_API_KEY (Anthropic mode enabled).")
else:
    print("Notice: No API key detected. Running with 'multilingual' clinical rule-based engine (100% offline, free, deterministic).")

cache_dir = PROCESSED_DIR / 'llm_raw'
cache_dir.mkdir(parents=True, exist_ok=True)
cached_studies = len(list(cache_dir.glob('*.json')))
total_studies = len(train_df)
print(f"Current progress: {cached_studies} / {total_studies} studies already cached ({cached_studies/max(total_studies, 1)*100:.1f}%).")
print("Any re-run will safely skip already processed studies without repeating extraction.")


In [ ]:
# --- Cell 9: Run Gemini / Clinical Re-labeling on Reports ---
if gemini_key:
    provider = "gemini"
    model_name = "gemini-3.8-flash"
    active_key = gemini_key
elif anthropic_key:
    provider = "anthropic"
    model_name = "claude-3-5-sonnet-20241022"
    active_key = anthropic_key
else:
    provider = "multilingual"
    model_name = "clinical-multilingual"
    active_key = None

print(f"Executing Phase 2 re-labeling with provider='{provider}', model='{model_name}'...")

# Set limit=None to process all studies, or e.g. limit=50 for a fast trial
labels_df, audit_df = relabel_all_studies(
    raw_dir=RAW_DIR,
    processed_dir=PROCESSED_DIR,
    provider=provider,
    model=model_name,
    api_key=active_key,
    limit=None,
)

print(f"✓ Generated train_labels_llm.csv: {len(labels_df)} studies fully labeled.")
print(f"✓ Generated label_audit.csv: {len(audit_df)} audit comparisons.")
if Path("clean_train.csv").exists():
    print("✓ Generated clean_train.csv in workspace root (0 missing labels).")


In [ ]:
# --- Cell 10: Inspect Reconciled Labels & Agreement ---
audit_path = PROCESSED_DIR / 'label_audit.csv'
if audit_path.exists():
    audit = pd.read_csv(audit_path)

    # Agreement rate per label for rows that had labels in train.csv
    rates = {}
    for label in TARGET_LABELS:
        orig_col = f"{label}_orig"
        pred_col = f"{label}_nlp" if f"{label}_nlp" in audit.columns else f"{label}_llm"
        if orig_col in audit.columns and pred_col in audit.columns:
            valid = audit[orig_col].notna()
            if valid.sum() > 0:
                rate = (audit.loc[valid, orig_col] == audit.loc[valid, pred_col]).mean() * 100
                rates[label] = rate

    if rates:
        plt.figure(figsize=(10, 6))
        plt.barh(list(rates.keys()), list(rates.values()), color='#1f77b4')
        plt.xlabel('Agreement Rate (%)')
        plt.title('Agreement Rate on Previously Labeled Studies')
        plt.xlim(0, 105)
        plt.grid(axis='x', linestyle='--', alpha=0.7)
        plt.tight_layout()
        plt.show()

    # Display sample newly labeled studies
    print("Sample reconstructed labels from reports:")
    display(labels_df.head(10))
else:
    print("label_audit.csv not found.")
